# Resumo e Decisões

## Parte 1:
Esses são os pontos que vou retirar, setembro de 2021, maio de 2024 e dezembro de 2025. Os dois primeiros baseados na média geral e o último baseado na média do respectivo mês nos últimos 4 anos

## Parte 2:
Removido os outliers e feita a média de geração de cada mês. No ponto de outlier foi alterado o valor para a média do mês nos 4 anos disponíveis. Dessa forma evitamos problemas na etapa seguinte com modelos que precisam de dados contínuos.

### Parte 1

Importar as bibliotecas e carregar os dados de geração

In [122]:
import pandas as pd

# 1. Carregar os dados
tabela_usina = pd.read_excel('power_plant_data_teste.xlsx')

# verificar se tem linhas vazias
display(tabela_usina.info())

# Mostrar a tabela
display(tabela_usina)

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  61 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    61 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 1.6 KB


None

,Geração Mensal Referência Month,Unidade Consumidora (UC) Usina (Nickname),Geração Mensal SUM Energia Gerada (kWh)
0,NaT,Usina Teste,0
1,2021-07-01,Usina Teste,374203
2,2021-08-01,Usina Teste,404653
3,2021-09-01,Usina Teste,426355
4,2021-10-01,Usina Teste,467200
...,...,...,...
56,2026-02-01,Usina Teste,438651
57,2026-03-01,Usina Teste,363221
58,2026-04-01,Usina Teste,364546
59,2026-05-01,Usina Teste,306272


In [123]:
#Temos uma linha vazia na tabela! Limpando essa linha
tabela_limpa = tabela_usina.dropna()
display(tabela_limpa.info())

display(tabela_limpa.describe().round(1))

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 1 to 60
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  60 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    60 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 1.5 KB


None

,Geração Mensal Referência Month,Geração Mensal SUM Energia Gerada (kWh)
count,60,60.0
mean,2023-12-16 11:12:00,417105.5
min,2021-07-01 00:00:00,110463.0
25%,2022-09-23 12:00:00,352583.5
50%,2023-12-16 12:00:00,422588.0
75%,2025-03-08 18:00:00,494157.2
max,2026-06-01 00:00:00,730252.0
std,NaN,101307.7


In [124]:
import plotly.express as px

# Gráfico de barras simples: Mês no eixo X e Geração no eixo Y
fig = px.bar(
    tabela_limpa, 
    x='Geração Mensal Referência Month', 
    y='Geração Mensal SUM Energia Gerada (kWh)',
    title='Geração Mensal da Usina (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()


### Importante:
    Temos uma média de geração de 417105 kWh no período avaliado
    Setembro de 2022 tem a maior geração com 730252 kWh, aproximadamente 75% mais produção que a media
    Maio de 2024 tem a menor geração com 110463 kWh, aproximadamente 26% da média
    Dezembro de 2025 está com geração de 266 kWh, sendo aproximadamente 50% da média dos últimos 4 anos (21 a 24 => média aproximada 519 kwh)

Esses são os pontos que vou modificar: setembro de 2022, maio de 2024 e dezembro de 2025. Os dois primeiros baseados na média geral e o último baseado na média do respectivo mês nos últimos 4 anos


### Parte 2

In [125]:
# variáveis auxiliares
col_data = 'Geração Mensal Referência Month'
col_geracao = 'Geração Mensal SUM Energia Gerada (kWh)'

# Converter a data
tabela_usina[col_data] = pd.to_datetime(tabela_usina[col_data])

# Definir os 3 pontos de outliers
datas_outliers = pd.to_datetime(['2022-09-01', '2024-05-01', '2025-12-01'])

# Criar uma cópia da tabela para correção
tabela_corrigida = tabela_limpa.copy()

# Criar coluna auxiliar com o mês do ano (1 a 12)
tabela_corrigida['mes_num'] = tabela_limpa[col_data].dt.month

# ver se temos todas as linhas
#display(tabela_limpa.info())
display(tabela_corrigida.info())

# Calcular a média do mês(excluindo os outliers)
dados_sem_outliers = tabela_corrigida[~tabela_corrigida[col_data].isin(datas_outliers)]
medias_historicas = dados_sem_outliers.groupby('mes_num')[col_geracao].mean()

# mostrar tabela
#display(tabela_corrigida)


<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 1 to 60
Data columns (total 4 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  60 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    60 non-null     int64         
 3   mes_num                                    60 non-null     int32         
dtypes: datetime64[us](1), int32(1), int64(1), str(1)
memory usage: 1.8 KB


None

In [127]:
# Criar a coluna tratada (inicialmente idêntica à original)
tabela_corrigida['geracao_corrigida'] = tabela_limpa[col_geracao].astype(float)
display(tabela_corrigida.info())

# Substituir o valor dos 3 outliers pela média dos seus respectivos meses
for data in datas_outliers:
    mes = data.month
    valor_media = medias_historicas[mes]
    tabela_corrigida.loc[tabela_limpa[col_data] == data, 'geracao_corrigida'] = valor_media

# display(tabela_corrigida.info())
# display(tabela_corrigida)

# Removendo a coluna auxiliar
tabela_corrigida = tabela_corrigida.drop(columns=['mes_num'])
# display(tabela_corrigida.info())
# display(tabela_corrigida)


# --- Relatório do Antes e Depois ---
print("=== RELATÓRIO DE CORREÇÃO (PARTE 2) ===")
filtro_outliers = tabela_limpa[col_data].isin(datas_outliers)
relatorio = tabela_corrigida.loc[filtro_outliers, [col_data, col_geracao, 'geracao_corrigida']]
relatorio.columns = ['Data', 'Valor Original (Outlier)', 'Valor Corrigido (Média dos 4 anos)']

display(relatorio)

# display(tabela_limpa.info())

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 1 to 60
Data columns (total 4 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  60 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    60 non-null     int64         
 3   geracao_corrigida                          60 non-null     float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 2.0 KB


None

KeyError: "['mes_num'] not found in axis"

In [6]:
# Plotando o gráficos antes e depois para verificar os dados removidos

# antes
fig = px.bar(
    tabela_usina, 
    x='Geração Mensal Referência Month', 
    y='Geração Mensal SUM Energia Gerada (kWh)',
    title='Geração Mensal da Usina antes (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()

# depois
fig = px.bar(
    tabela_limpa, 
    x='Geração Mensal Referência Month', 
    y='geracao_corrigida',
    title='Geração Mensal da Usina depois (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()

Foram corrigidos os dados referentes a setembro de 2022, maio de 2024 e dezembro de 2025. Para todos os casos foram modificados seus valores para a média dos outros 4 meses disponíveis.